# Summary plots — Az-consistent-forward scalar-amplitude inversion (CK + non-CK)

Counterpart to `plt_syn_slip_az_inv_hetmu_nicoya_locking_summary.ipynb` for the
new cf scheme.  Loads BOTH plate models side by side:

* **CK** plate model → mesh `nicoyaCKden_une_sm`, projection `LL2ckmd` (rot=45°, offset (130,350) km)
* **non-CK** flat plate model → mesh `nicoyaden_sm`, projection `azi_equidist_proj` (no rotation)

For each plate model, summary panels show the recovery ratio
(`rec_amp / amp`) under all 5 forward/inversion combinations:
`(homhom, swhom, swsw, 3dhom, 3d3d)`.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/synth/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
flag_savefig = True
# flag_savefig = False

# flag to indicate the inversion scheme
inv_scheme = "_az"   # to indicate that the inversion fixed the azimuth of slip

# Define the inversion results path
resultpath = "syn_slip/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshname = "nicoyaCK4"   # same as CK2, but connecting the trench nowprint(meshname)

# Meshes with even top boundary at 0 depth
# meshname = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshname = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
meshname = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
# meshname = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshname

print(meshname)

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
# print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
# print(trench.head())

# anaysis type, locking or coseismic?
type = "locking"  # "locking" or "coseismic"
# type = "coseismic"

# whethere the synthetics are polluted
pollute = True  # True or False
# pollute = False
print(pollute)

# noise std type, either 'uniform' or 'datastd'
pollute_type = 'uniform'  # uniform noise for all stations
# pollute_type = 'datastd'  # use the data standard deviation as noise std
print(pollute_type)

m2mm = 1e3  # meter to mm
km2m = 1e3   # km to m

if type == "locking":
       # read in the lon and lat of stations
       # obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
       obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

       # note that the height is in m, Dt in years, the original displacement data is in mm/yr
       # the disp relative to the stable Caribbean plate will be used in the inversion
       # From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                            'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                            'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
       df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
       # Convert mm to m, needed for inversion
       cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                     'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
       df[cols_to_convert] = df[cols_to_convert] / m2mm  # Convert mm to m

       # displacement noise standard deviations, in m 
       if pollute:
              if pollute_type == 'uniform':
                     noise_std_h = 0.5 * (df['vx_std_Car'].mean() + df['vy_std_Car'].mean()) 
                     noise_std_v = df['vz_std_Car'].mean()
                     error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v
                     
              elif pollute_type == 'datastd':
                     error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
                     
       else:
              error_e, error_n, error_v = df['vx_std_Car']*0, df['vy_std_Car']*0, df['vz_std_Car']*0

else:
       obs_file = "data/Protti_etal_2014_tableS1.csv"
       # note that the height is in m, duration and dates are in yr, and the displacements and errors are already in m
       # From BAGA to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                            'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])
       
       # displacement noise standard deviations
       if pollute:
              noise_std_h = 0.5 * (df['ux_std'].mean() + df['uy_std'].mean())
              error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_h
       else:
              error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())

In [ ]:
# === Az-consistent-forward scheme: bounded inversion is the only mode ===
V_para = 16.0 / m2mm
V_norm = 78.5 / m2mm
print(f"V_norm = {V_norm*m2mm:.1f} mm/yr (trench-normal long-term loading)")


In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
# print(volcano.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
region1=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]

In [ ]:
### Choose the slip model, all models are in m_dip direction, and m_strike = 0
### All are offseted a little along dip due to the plate shape from CK
# slipmodel = 1     # along-strike 3-stripe pattern, shallow-middle-deep
slipmodel = 2     # along-strike 2-stripe pattern with gap, shallow-deep 
# slipmodel = 3     # along-strike 1-stripe pattern, complementary of model 2, middle
# slipmodel = 4     # checkerboard pattern in strike-dip direction
# slipmodel = 5     # checkerboard pattern in x and y direction  
# slipmodel = 6     # same as model 4, but amp up to 1 m rather than V_norm=78.5 mm
# slipmodel = 7     # 2D Gaussian where the contours are circular, within range of 0 and max

# ============================================================
# AZIMUTH SLIP PRESCRIPTION PARAMETERS
# slip direction fixed to a geographic azimuth (N45E = plate convergence)
# rather than averaged local down-dip direction
# ============================================================
# slip_azimuth_deg  = 45.0   # CW from North; N45E = Cocos-Caribbean trench-normal convergence
# slip_azimuth_deg  = 26.0   # CW from North; N26E = roughly is oblique convergence
slip_azimuth_deg  = 33.5   # CW from North; N33.5E, a little oblique convergence

V_norm = 78.5   # in mm
V_para = 27
V_para_const = 11

if slip_azimuth_deg == 45.0:
    amp = V_norm  # 78.5 mm
elif slip_azimuth_deg == 26.0:
    amp = round(np.sqrt(V_norm**2+V_para**2))  # about 83 mm
elif slip_azimuth_deg == 33.5:
    amp = round(np.sqrt(V_norm**2+(V_para-V_para_const)**2))  # about 80 mm

amp = amp / 1e3     # convert mm to m

if slipmodel == 2:
    # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
    # amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 80e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 35e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = -45e3     # y center of the pattern
    pattern_rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    # y_len = 240e3   # length of each rectangle in strike direction  
    # dx = 40e3  # gap between rectangles
    # y0 = -40e3     # y center of the pattern
    # zmin = -70e3
    # zmax = 0

    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 4:
    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    # amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    pattern_rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    # slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{pattern_rot_deg:g}_ms{amp:g}"    
    slip_str_gt = (
        f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )
    
slip_str_gt1 = slip_str_gt  #with no pollution

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt = slip_str_gt + "_pou"
    
    elif pollute_type == 'datastd':
        slip_str_gt = slip_str_gt + "_pod"

print("Slip model string:", slip_str_gt)

# === Az-scheme amp from plate-convergence azimuth (cf scripts use this convention) ===
slip_azimuth_deg = 33.5
V_norm_mm = 78.5
V_para = 27
V_para_const = 11
if slip_azimuth_deg == 45.0:
    amp = V_norm_mm
elif slip_azimuth_deg == 26.0:
    amp = round(np.sqrt(V_norm_mm**2 + V_para**2))
elif slip_azimuth_deg == 33.5:
    amp = round(np.sqrt(V_norm_mm**2 + (V_para - V_para_const)**2))
amp = amp / m2mm
print(f"amp = {amp*m2mm:.1f} mm/yr  (slip_azimuth_deg = {slip_azimuth_deg})")

# === Derive slip_pattern + pattern_option ===
if slipmodel in (1, 2, 3):
    slip_pattern = "stripe"
    pattern_option = {1: 0, 2: 1, 3: 2}[slipmodel]
elif slipmodel in (4, 5, 6):
    slip_pattern = "checker"
    pattern_option = {4: 1, 5: 2, 6: 3}[slipmodel]
else:
    slip_pattern = "other"
    pattern_option = 0
print("slip_pattern:", slip_pattern, " pattern_option:", pattern_option)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # string for the contrast between slab and wedge
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the original 3d model
    het3d_str_ori = f"_DeShon3D_ref1D_{round(1.0)}_hull"
    # string for the 3d model, value multiplied by 4, relative a reference
    # contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
    contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026
    if not use_uneven_mesh:
        het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"
    else:
        # het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
        het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
        # het3d_str_4 = f"_DeShon3D_ref1D_{round(4.0)}_hull"

        CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
        if CG_mu_deg == 2:
            het3d_str = het3d_str + f"_CG{CG_mu_deg}"
            het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(sw_str)
print(het3d_str_ori)
print(het3d_str)

In [ ]:
# Define the reference point (center of the projection) — CK convention
lon0, lat0 = -84, 7
rot = 45
x0, y0 = 130e3, 350e3

# Load the CK fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1,
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f / km2m
loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)

# ── Ground-truth slip from cf forward (slip_true_cf_*.txt) ───────────────────
# Columns: xf yf zf m_phys s_strike s_dip slip_mag c_mag (m, m/yr)
# m_phys is the prescribed scalar amplitude in m/yr (= amp on locked vertices, 0 on creeping)
outFileName = 'slip_true_cf_' + meshname + slip_str_gt + '.txt'
print(outFileName)
slip_cf_raw = np.loadtxt(datadir + resultpath + outFileName, comments='#')
mtrue_s = pd.DataFrame({
    's_strike': slip_cf_raw[:, 4],   # m/yr (along-fault projected)
    's_dip':    slip_cf_raw[:, 5],   # m/yr
    'mag':      slip_cf_raw[:, 3],   # m_phys, m/yr
})
print('Raw s_strike range (m/yr):', mtrue_s['s_strike'].min(), mtrue_s['s_strike'].max())
print('Raw s_dip range (m/yr):', mtrue_s['s_dip'].min(), mtrue_s['s_dip'].max())
print('Raw m_phys range (m/yr):', mtrue_s['mag'].min(), mtrue_s['mag'].max())

# Normalize to 0..1 (plotting code multiplies by amp*m2mm to get mm where needed)
if type == 'locking':
    cols_to_convert = ['s_strike', 's_dip', 'mag']
    mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] / amp
print('Normalized mag (locking ratio) range:', mtrue_s['mag'].min(), mtrue_s['mag'].max())


In [ ]:
# Coordinate rotation (CK side uses rot=45)
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot =  x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot
    return x_rot, y_rot

# ── Read ground-truth moment / mag / potency from cf forward ──────────────────
def read_gt_moment(datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'moment_true_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    return pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                       names=['moment', 'mw', 'potency'])

# ── Read synthetic station displacements (cf forward) — CK rotation ──────────
def read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_true['lon'], u_true['lat'] = df['lon'], df['lat']
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot)
    u_true['mag']   = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true

def read_station_grid(datadir, resultpath, meshname, grid_spacing_deg=None):
    if grid_spacing_deg is None:
        outFileName = 'dense_grid_coordinates_' + meshname + '.txt'
    else:
        outFileName = f"dense_grid_coordinates_{meshname}_{grid_spacing_deg}.txt"
    print(outFileName)
    sta_grid = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                           names=['lon', 'lat', 'x', 'y', 'z'])
    sta_grid[['x', 'y', 'z']] = sta_grid[['x', 'y', 'z']] * km2m
    return sta_grid

def read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt,
                     struc_str_for, grid_spacing_deg=None):
    if grid_spacing_deg is None:
        outFileName = 'd_obs_grid_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    else:
        outFileName = ('d_obs_grid_cf_' + meshname + slip_str_gt
                       + struc_str_for + str(grid_spacing_deg) + '.txt')
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_true['lon'], u_true['lat'] = sta_grid['lon'], sta_grid['lat']
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot)
    u_true['mag']   = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true

def build_diff_disp(u_1, u_2):
    u_diff = u_1.copy()
    for c in ['ux', 'uy', 'uz', 'mag', 'mag_h', 'mag_v']:
        u_diff[c] = u_2[c] - u_1[c]
    return u_diff

def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    df_obs_h_data = {
        "x": u_true['lon'], "y": u_true['lat'],
        "east_velocity":  u_true['ux']*scaleunit,
        "north_velocity": u_true['uy']*scaleunit,
    }
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"]  = error_e*scaleunit
        df_obs_h_data["north_sigma"] = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0
    df_obs_h = pd.DataFrame(data=df_obs_h_data)
    df_obs_v_data = {
        "x": u_true['lon'], "y": u_true['lat'],
        "east_velocity": 0.0, "north_velocity": u_true['uz']*scaleunit,
    }
    if error_v is not None:
        df_obs_v_data["east_sigma"]  = error_v*scaleunit
        df_obs_v_data["north_sigma"] = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    return df_obs_h, df_obs_v

def build_disp_vector_grid(u_true, scaleunit, inc):
    unique_lons = u_true['lon'].unique()
    unique_lats = u_true['lat'].unique()
    n_lon, n_lat = len(unique_lons), len(unique_lats)
    print(f"Detected grid: {n_lat} x {n_lon}")
    ux_2d = u_true['ux'].to_numpy().reshape(n_lat, n_lon)
    uy_2d = u_true['uy'].to_numpy().reshape(n_lat, n_lon)
    lon_2d = u_true['lon'].to_numpy().reshape(n_lat, n_lon)
    lat_2d = u_true['lat'].to_numpy().reshape(n_lat, n_lon)
    ux_sub = ux_2d[::inc, ::inc]; uy_sub = uy_2d[::inc, ::inc]
    lon_sub = lon_2d[::inc, ::inc]; lat_sub = lat_2d[::inc, ::inc]
    ux = ux_sub.flatten(); uy = uy_sub.flatten()
    angle = np.degrees(np.arctan2(uy, ux))
    length = np.hypot(ux, uy)
    df_obs_h_velo = pd.DataFrame(data={
        "x": lon_sub.flatten(), "y": lat_sub.flatten(),
        "east_velocity": ux*scaleunit, "north_velocity": uy*scaleunit,
    })
    df_obs_h_plot = pd.DataFrame({
        'x': lon_sub.flatten(), 'y': lat_sub.flatten(),
        'angle': angle, 'length': length*scaleunit,
    })
    return df_obs_h_velo, df_obs_h_plot


In [ ]:
# read in the observation disp. for various structure models, HOMgeneous model
u_true_hom = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, homo_str)
# build observation disp. vectors for various structure models, HOMgeneous model
df_obs_h_hom, df_obs_v_hom = build_disp_vector(u_true_hom, m2mm, error_e, error_n, error_v)
# read in the ground-turth moment, mag, and potency, HOMgeneous model
mom_true_hom = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, homo_str)

# Slab/Wedge model
u_true_sw = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, sw_str)
df_obs_h_sw, df_obs_v_sw = build_disp_vector(u_true_sw, m2mm, error_e, error_n, error_v)
mom_true_sw = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, sw_str)
# build disp difference between various structure models and homogeneous
u_true_swdiff = build_diff_disp(u_true_hom, u_true_sw)
# build differential observation disp. vectors for various structure models wrt homogeneous
df_obs_h_swdiff, df_obs_v_swdiff = build_disp_vector(u_true_swdiff, m2mm, error_e, error_n, error_v)

### OLD, 3D DeShon model, value multiplied by 4, relative to a reference
# NEW as of 03/05/2026, 3D DeShon model, value multiplied by 2.5, relative to a reference 1D 
u_true_3d = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str)
df_obs_h_3d, df_obs_v_3d = build_disp_vector(u_true_3d, m2mm, error_e, error_n, error_v)
mom_true_3d = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str)

u_true_3ddiff = build_diff_disp(u_true_hom, u_true_3d)
df_obs_h_3ddiff, df_obs_v_3ddiff = build_disp_vector(u_true_3ddiff, m2mm, error_e, error_n, error_v)

In [ ]:
### For testing, the GT potency for diff structure models should be the same
print(mom_true_hom['potency'].to_numpy())
print(mom_true_sw['potency'].to_numpy())
print(mom_true_3d['potency'].to_numpy())

In [ ]:
# Decide the weights of the horizontal, vertical components
if pollute:
    if pollute_type == 'uniform':   
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1 
        # f_h, f_v = 1, 1/2 
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/noise_std_h), int(1/noise_std_v)
        
    elif pollute_type == 'datastd':
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1
        # f_h, f_v = 1, 1/2
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/f_h), int(1/f_v)
else:
    # Decide the weights of the horizontal, vertical components
    f_h, f_v = 1, 1
    # Print the weights of the data
    print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )

    # Take the inverse for saving the name of the weights
    w_h, w_v = int(1/f_h), int(1/f_v)


In [ ]:
# Define regularization weights (CK side)
rho_s = 1e9
gamma_val_H1 = 1.5e2     # current preferred under cf scheme
# gamma_val_H1 = 1.8e2
delta_val_L2 = gamma_val_H1 / rho_s
print(f"CK: gamma_val_H1 = {gamma_val_H1:.1e}, delta_val_L2 = {delta_val_L2:.1e}")

inv_str = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
print(inv_str)

print("slip model number:", slipmodel)


In [ ]:
# ── Load predicted moment / magnitude / potency ───────────────────────────────
def read_pred_moment(datadir, resultpath, meshname, slip_str_gt,
                     struc_str_for, inv_str, struc_str_inv):
    outFileName = 'moment_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    return pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                       names=['moment', 'mw', 'potency'])

# ── Load predicted station displacement (cf inversion, CK rotation) ───────────
def read_pred_disp(u_true, datadir, resultpath, meshname, slip_str_gt,
                   struc_str_for, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot)
    u_res = u_pred.copy()
    u_res['ux'] = u_true['ux'] - u_pred['ux']
    u_res['uy'] = u_true['uy'] - u_pred['uy']
    u_res['uz'] = u_true['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    return u_pred, u_res

# ── Load recovered amplitude + recovery ratio (cf inversion) — projection-independent ──
def read_inferred_slip(datadir, resultpath, meshname, slip_str_gt,
                       struc_str_for, inv_str, struc_str_inv, type):
    outFileName = ('slip_recovery_' + meshname + slip_str_gt
                   + struc_str_for + inv_str + struc_str_inv + '.txt')
    print(outFileName)
    raw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', comment='#', header=None)
    if raw.shape[1] == 7:
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio_global','ratio_local']
        m_s = pd.DataFrame({'mag': raw['ratio_global'].values})
    else:
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio']
        m_s = pd.DataFrame({'mag': raw['ratio'].values})
    m_s['s_strike'] = 0.0
    m_s['s_dip']    = m_s['mag']
    print(f"Recovered locking ratio range: [{m_s['mag'].min():.3f}, {m_s['mag'].max():.3f}]")
    return m_s


In [ ]:
### Load the predicted surface displacements and residuals for diff forward and inversion models

## hom forward, hom inversion
u_pred_homhom, u_res_homhom = read_pred_disp(u_true_hom, datadir, resultpath, meshname, slip_str_gt, \
                                             homo_str, inv_str, homo_str)
m_s_homhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                homo_str, inv_str, homo_str, type)
mom_homhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, homo_str, inv_str, homo_str)

## slab/wedge forward, hom inversion
u_pred_swhom, u_res_swhom = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, homo_str)
m_s_swhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, homo_str, type)
mom_swhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, homo_str)

## slab/wedge forward, slab/wedge inversion
u_pred_swsw, u_res_swsw = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, sw_str)
m_s_swsw = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, sw_str, type)
mom_swsw = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, sw_str)
# difference between the heterogeneous and homogeneous model
print((m_s_swsw['s_dip']-m_s_swhom['s_dip']).min(), (m_s_swsw['s_dip']-m_s_swhom['s_dip']).max())
# print((m_s_swsw['mag']-m_s_swhom['mag']).min(), (m_s_swsw['mag']-m_s_swhom['mag']).max())


## 3d forward, hom inversion
u_pred_3dhom, u_res_3dhom = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, homo_str)
m_s_3dhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, homo_str)

## amplified 3d forward, 3d inversion
u_pred_3d3d, u_res_3d3d = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, het3d_str)
m_s_3d3d = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, het3d_str)
# difference between the heterogeneous and homogeneous model
print((m_s_3d3d['s_dip']-m_s_3dhom['s_dip']).min(), (m_s_3d3d['s_dip']-m_s_3dhom['s_dip']).max())
# print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())


In [ ]:
# assess_col = 's_dip'   # assessment component
assess_col = 'mag'   # assessment component

## Load all related results when using the flat plate interface slab2.0

In [ ]:
# === Non-CK plate model setup (mesh nicoyaden_sm, no rotation/offset) ===
meshname_flat = "nicoyaden_sm"
print(meshname_flat)

# Plate / trench setup (kept from old notebook)
plate_file_flat = "plateinterface/nicoya_slab2_100_10_10.txt"
df_plate_flat = utp.parse_plate_interface_file(datadir + plate_file_flat)
trench_file_flat = "plateinterface/trench.gmt.inuse"
# Plate interface in GMT .grd format (used directly by grdcontour, non-CK idiom)
grd_file_flat = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"
segments = utp.parse_trench_file(datadir + trench_file_flat,
                                 lon_range=(region[0], region[1]),
                                 lat_range=(region[2], region[3]))
trench_flat = utp.segments_to_dataframe(segments)

# Non-CK slip_str_gt  (note: x_len=90 km, y0=-40 km — different from CK x_len=80, y0=-45)
if slipmodel == 2:
    x_len = 90e3; y_len = 300e3; dx = 35e3
    x0_p = 0; y0_p = -40e3
    pattern_rot_deg = 0.0
    slip_str_gt_flat = (
        f"_stripe_x{x0_p/1e3:g}_y{y0_p/1e3:g}"
        f"_lx{x_len/1e3:g}_dx{dx/1e3:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )
elif slipmodel == 4:
    dx = 35e3; dy = 35e3; x0_p = -20e3; y0_p = -45e3; pattern_rot_deg = 45
    slip_str_gt_flat = (
        f"_check_x{x0_p/km2m:g}_y{y0_p/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt_flat = slip_str_gt_flat + "_pou"
    elif pollute_type == 'datastd':
        slip_str_gt_flat = slip_str_gt_flat + "_pod"
print("Non-CK slip model string:", slip_str_gt_flat)

# Non-CK projection: azi_equidist_proj, no rotation, no offset
lon0_flat, lat0_flat = -85.5, 10
R = 6371.0

# Non-CK fault geometry
outFileName = 'fault_geometry_' + meshname_flat + '.txt'
loc_f_flat = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1,
                         names=['xf', 'yf', 'zf'])
loc_f_flat = loc_f_flat / km2m
loc_f_flat['lon'], loc_f_flat['lat'] = utp.inverse_azi_equidist_proj(
    loc_f_flat['xf'], loc_f_flat['yf'], lon0_flat, lat0_flat)

# Non-CK ground-truth slip from cf forward
outFileName = 'slip_true_cf_' + meshname_flat + slip_str_gt_flat + '.txt'
print(outFileName)
slip_cf_raw_flat = np.loadtxt(datadir + resultpath + outFileName, comments='#')
mtrue_s_flat = pd.DataFrame({
    's_strike': slip_cf_raw_flat[:, 4],
    's_dip':    slip_cf_raw_flat[:, 5],
})
mtrue_s_flat['mag'] = slip_cf_raw_flat[:, 3] / amp
print("Non-CK true locking ratio range:", mtrue_s_flat['mag'].min(), mtrue_s_flat['mag'].max())

# Non-CK cf inv_str  (same regularization choices as CK side)
rho_s_flat = 1e9
gamma_val_H1_flat = 1.5e2
# gamma_val_H1_flat = 1.8e2
delta_val_L2_flat = gamma_val_H1_flat / rho_s_flat
inv_str_flat = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1_flat:.1e}_ds{delta_val_L2_flat:.1e}"
print("Non-CK", inv_str_flat)

# Load all 5 inversion combos for non-CK (same model matrix as CK)
m_s_homhom_flat = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                     homo_str, inv_str_flat, homo_str, type)
mom_homhom_flat = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                   homo_str, inv_str_flat, homo_str)

m_s_swhom_flat  = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                     sw_str, inv_str_flat, homo_str, type)
mom_swhom_flat  = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                   sw_str, inv_str_flat, homo_str)

m_s_swsw_flat   = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                     sw_str, inv_str_flat, sw_str, type)
mom_swsw_flat   = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                   sw_str, inv_str_flat, sw_str)

m_s_3dhom_flat  = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                     het3d_str, inv_str_flat, homo_str, type)
mom_3dhom_flat  = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                   het3d_str, inv_str_flat, homo_str)

m_s_3d3d_flat   = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                     het3d_str, inv_str_flat, het3d_str, type)
mom_3d3d_flat   = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                   het3d_str, inv_str_flat, het3d_str)


In [ ]:
# colorlabel = "Locking difference"
colorlabel = "Difference in locking ratio"


In [ ]:
def plot_sw_3d_hom_slip_diff_summary(m_s_hom, m_s_sw, m_s_3d, m_s_hom_flat, m_s_sw_flat, m_s_3d_flat, 
                             col2plt, region, maxdiff_plot=None, diff_step=None, flag_savefig=False):
    
    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    # colormap = "viridis"
    # colormap = "rainbow"
    colormap = "roma"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.8c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # mu_model_labels = ["S_S - H_H", "3D_3D - H_H", "S_S - H_H", "3D_3D - H_H"]
        # mu_model_labels = ["S - H", "3D - H", "S - H", "3D - H"]
        mu_model_labels = ["S", "3D", "S", "3D"]
        # plate_labels = ["Rough", "Smooth"]
        plate_labels = ["Irregular", "Regular"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        #### Results from using Rough plate interface slab2 ####
    
        # Difference between the SW and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            print((m_s_sw[col2plt] - m_s_hom[col2plt]).min(), (m_s_sw[col2plt] - m_s_hom[col2plt]).max())
            maxdiff = np.max([(m_s_sw[col2plt] - m_s_hom[col2plt]).abs().max(), (m_s_3d[col2plt] - m_s_hom[col2plt]).abs().max()])
            maxdiff = (m_s_sw[col2plt] - m_s_hom[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            # diff_step = 0.02
            # diff_step = maxdiff/10
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = 0.5
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s_sw[col2plt] - m_s_hom[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_hom[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)            
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                
            #plot panel label
            fig.text(text=panel_labels[0], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")

        # Difference between the 3D and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[1]}"])
            print((m_s_3d[col2plt] - m_s_hom[col2plt]).min(), (m_s_3d[col2plt] - m_s_hom[col2plt]).max())
            maxdiff = (m_s_3d[col2plt] - m_s_hom[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)        
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s_3d[col2plt] - m_s_hom[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
        
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") 
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[1], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")


        #### Results from using Smooth plate interface slab2 ####
        slipsybsz = "c0.125c"

        # difference between the SW and homogeneous model
        with fig.set_panel(panel=[1, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[2]}"])
            print((m_s_sw_flat[col2plt] - m_s_hom_flat[col2plt]).min(), (m_s_sw_flat[col2plt] - m_s_hom_flat[col2plt]).max())
            maxdiff = np.max([(m_s_sw[col2plt] - m_s_hom[col2plt]).abs().max(), (m_s_3d[col2plt] - m_s_hom[col2plt]).abs().max()])
            maxdiff = (m_s_sw_flat[col2plt] - m_s_hom_flat[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = 0.5
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s_sw_flat[col2plt] - m_s_hom_flat[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f_flat["lon"].iloc[order], y=loc_f_flat["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f_flat['lon'], y=loc_f_flat['lat'], style=slipsybsz, fill=m_s_sw_flat[col2plt] - m_s_hom_flat[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_hom[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)            
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file_flat, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench_flat['lon'], y=trench_flat['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") 
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[2], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[1], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")


        # ABS difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[3]}"])
            print((m_s_3d_flat[col2plt] - m_s_hom_flat[col2plt]).min(), (m_s_3d_flat[col2plt] - m_s_hom_flat[col2plt]).max())
            maxdiff = (m_s_3d_flat[col2plt] - m_s_hom_flat[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)        
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_diff = m_s_3d_flat[col2plt] - m_s_hom_flat[col2plt]
            order = m_s_diff.argsort()
            fig.plot(x=loc_f_flat["lon"].iloc[order], y=loc_f_flat["lat"].iloc[order], style=slipsybsz, fill=m_s_diff.iloc[order], cmap=True)
            # fig.plot(x=loc_f_flat['lon'], y=loc_f_flat['lat'], style=slipsybsz, fill=m_s_3d_flat[col2plt] - m_s_hom_flat[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_diff.min():.2f}, {m_s_diff.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
        
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            fig.grdcontour(grid=grd_file_flat, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench_flat['lon'], y=trench_flat['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") 
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[3], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[1], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")

    fig.show()

    if flag_savefig:
        # figname = meshname + "_slip" + str(slipmodel) + "_sum_invslip_diff_re_hom_roughvsflat.pdf"
        figname = meshname + inv_scheme + "_slip" + str(slipmodel) + "_ref1D_sum_invslip_diff_re_hom_roughvsflat.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

    # return fig

## difference relative to the homo model
# maxdiff_plot = 0.18
maxdiff_plot = 0.05     # NEW 3D model, x2.5 relative to 1D reference
# diff_step = 0.02
diff_step = maxdiff_plot/10

plot_sw_3d_hom_slip_diff_summary(m_s_homhom, m_s_swsw, m_s_3d3d, 
                                 m_s_homhom_flat, m_s_swsw_flat, m_s_3d3d_flat, 
                                 assess_col, region1, maxdiff_plot, diff_step, flag_savefig=flag_savefig)


# Under the same observation (displacements) from a heterogeneous structure, how does inversion vary if ignoring the structure?

* In other words, the inversion is based on the same synthetic data, which is generated from the respective heterogeneous structure (either Slab/Wedge model or amplified 3D model)


In [ ]:
# Chosen γ for cf scheme (CK side) — used for the heterogeneous-vs-hom inversion plots
rho_s = 1e9
gamma_val_H1 = 1.5e2
# gamma_val_H1 = 1.8e2
delta_val_L2 = gamma_val_H1 / rho_s
print(f"CK chosen γ: {gamma_val_H1:.1e}")

inv_str = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
print(inv_str)


In [ ]:
## slab/wedge forward, hom inversion
u_pred_swhom1, u_res_swhom1 = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, homo_str)
m_s_swhom1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, homo_str, type)
mom_swhom1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, homo_str)

## slab/wedge forward, slab/wedge inversion
u_pred_swsw1, u_res_swsw1 = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, sw_str)
m_s_swsw1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, sw_str, type)
mom_swsw1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, sw_str)
# difference between the heterogeneous and homogeneous model
print((m_s_swsw1['s_dip']-m_s_swhom1['s_dip']).min(), (m_s_swsw1['s_dip']-m_s_swhom1['s_dip']).max())
# print((m_s_swsw['mag']-m_s_swhom['mag']).min(), (m_s_swsw['mag']-m_s_swhom['mag']).max())


## 3d forward, hom inversion
u_pred_3dhom1, u_res_3dhom1 = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, homo_str)
m_s_3dhom1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, homo_str)

## 3d forward, 3d inversion
u_pred_3d3d1, u_res_3d3d1 = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, het3d_str)
m_s_3d3d1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, het3d_str)
# difference between the heterogeneous and homogeneous model
print((m_s_3d3d1['s_dip']-m_s_3dhom1['s_dip']).min(), (m_s_3d3d1['s_dip']-m_s_3dhom1['s_dip']).max())
# print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())


## Load all related results when using the flat plate interface slab2.0

In [ ]:
# Chosen γ for cf scheme (non-CK side)
rho_s_flat = 1e9
gamma_val_H1_flat = 1.5e2
# gamma_val_H1_flat = 1.8e2
delta_val_L2_flat = gamma_val_H1_flat / rho_s_flat
print(f"Non-CK chosen γ: {gamma_val_H1_flat:.1e}")

inv_str_flat = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1_flat:.1e}_ds{delta_val_L2_flat:.1e}"
print(inv_str_flat)

## Load non-CK het-forward / hom-inversion data (for cell 24 plot_homvshet_slip_diff_summary)
## slab/wedge forward, hom inversion (non-CK)
m_s_swhom1_flat  = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                      sw_str, inv_str_flat, homo_str, type)
mom_swhom1_flat  = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                    sw_str, inv_str_flat, homo_str)

## slab/wedge forward, slab/wedge inversion (non-CK)
m_s_swsw1_flat   = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                      sw_str, inv_str_flat, sw_str, type)
mom_swsw1_flat   = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                    sw_str, inv_str_flat, sw_str)

## 3d forward, hom inversion (non-CK)
m_s_3dhom1_flat  = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                      het3d_str, inv_str_flat, homo_str, type)
mom_3dhom1_flat  = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                    het3d_str, inv_str_flat, homo_str)

## 3d forward, 3d inversion (non-CK)
m_s_3d3d1_flat   = read_inferred_slip(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                      het3d_str, inv_str_flat, het3d_str, type)
mom_3d3d1_flat   = read_pred_moment(datadir, resultpath, meshname_flat, slip_str_gt_flat,
                                    het3d_str, inv_str_flat, het3d_str)
print((m_s_swsw1_flat['mag'] - m_s_swhom1_flat['mag']).min(),
      (m_s_swsw1_flat['mag'] - m_s_swhom1_flat['mag']).max())
print((m_s_3d3d1_flat['mag'] - m_s_3dhom1_flat['mag']).min(),
      (m_s_3d3d1_flat['mag'] - m_s_3dhom1_flat['mag']).max())


In [ ]:
def plot_homvshet_slip_diff_summary(m_s_swhom, m_s_swsw, m_s_3dhom, m_s_3d3d, 
                                    m_s_swhom_flat, m_s_swsw_flat, m_s_3dhom_flat, m_s_3d3d_flat, 
                                    col2plt, region, maxdiff=None, diff_step=None, flag_savefig=False):


    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    # colormap = "viridis"
    colormap = "roma"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.8c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # mu_model_labels = ["S_S - S_H", "3D_3D - 3D_H", "S_S - S_H", "3D_3D - 3D_H"]
        mu_model_labels = ["S - H", "3D - H", "S - H", "3D - H"]
        # plate_labels = ["Rough", "Smooth"]
        plate_labels = ["Irregular", "Regular"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        # Difference between the SW and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            m_s_plot = m_s_swsw[col2plt] - m_s_swhom[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # diff_step = 0.02
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.2p, darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[0], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")


        # Difference between the 3D and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            m_s_plot = m_s_3d3d[col2plt] - m_s_3dhom[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[1]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.2p, darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[1], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")   


        #### Results from using Smooth plate interface slab2 ####
        slipsybsz = "c0.125c"

        # Difference between the SW and homogeneous model
        with fig.set_panel(panel=[1, 0]):
            m_s_plot = m_s_swsw_flat[col2plt] - m_s_swhom_flat[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[2]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f_flat["lon"].iloc[order], y=loc_f_flat["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f_flat['lon'], y=loc_f_flat['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            fig.grdcontour(grid=grd_file_flat, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench_flat['lon'], y=trench_flat['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[2], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[1], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")   

        # Difference between the 3D and homogeneous model
        with fig.set_panel(panel=[1, 1]):
            m_s_plot = m_s_3d3d_flat[col2plt] - m_s_3dhom_flat[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[3]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            pygmt.makecpt(cmap=colormap, series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f_flat["lon"].iloc[order], y=loc_f_flat["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f_flat['lon'], y=loc_f_flat['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            fig.grdcontour(grid=grd_file_flat, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench_flat['lon'], y=trench_flat['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                # fig.colorbar(frame=["a0.1f0.02", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          
                fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[3], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")

            fig.text(text=plate_labels[1], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50") 
       

    fig.show()

    if flag_savefig:
        # figname = meshname + "_slip" + str(slipmodel) + "_sum_invslip_hethom_diff_roughvsflat.pdf"
        figname = meshname + inv_scheme + "_slip" + str(slipmodel) + "_ref1D_sum_invslip_hethom_diff_roughvsflat.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)


# maxdiff = (m_s_plot).abs().max()

maxdiff1 = np.max( [(m_s_swsw1[assess_col] - m_s_swhom1[assess_col]).abs().max(), 
                   (m_s_3d3d1[assess_col] - m_s_3dhom1[assess_col]).abs().max()]) 
maxdiff2 = np.max( [(m_s_swsw1_flat[assess_col] - m_s_swhom1_flat[assess_col]).abs().max(), 
                   (m_s_3d3d1_flat[assess_col] - m_s_3dhom1_flat[assess_col]).abs().max()]) 
print(maxdiff1, maxdiff2)

# maxdiff = 0.32
# maxdiff = 0.22      # NEW 3D model, x2.5 relative to 1D reference
maxdiff = 0.20      # NEW 3D model, x2.5 relative to 1D reference
# diff_step = 0.02
diff_step = maxdiff/10

plot_homvshet_slip_diff_summary(m_s_swhom1, m_s_swsw1, m_s_3dhom1, m_s_3d3d1, 
                                m_s_swhom1_flat, m_s_swsw1_flat, 
                                m_s_3dhom1_flat, m_s_3d3d1_flat,
                                assess_col, region1, maxdiff, diff_step, flag_savefig=flag_savefig)        
